# 01 Data validation

Validate the public processed table before any analysis. The checks deliberately recompute totals from county-service records rather than trusting source summary cells.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.portfolio_analysis import load_data

DATA_PATH = ROOT / 'data' / 'processed' / 'ecosystem_values_2018_2023.csv'
data = load_data(DATA_PATH)
data.head()


,year,county,county_zh,service,service_label,service_zh,value_yuan,value_cny_100m
0,2018,Lushi County,卢氏,provisioning,Provisioning,供给,1.483245e+08,1.483245
1,2018,Lushi County,卢氏,water_retention,Water retention,水源,4.501880e+07,0.450188
2,2018,Lushi County,卢氏,carbon_storage,Carbon storage,碳储,1.394797e+09,13.947970
3,2018,Lushi County,卢氏,soil_retention,Soil retention,土保,4.041572e+09,40.415718
4,2018,Lushi County,卢氏,recreation,Recreation,旅游,4.243289e+09,42.432886


In [2]:
from src.portfolio_analysis import regional_totals, validate_data

checks = validate_data(data)
checks


{'rows': 110,
 'counties': 11,
 'services': 5,
 'total_2018_cny_100m': 2201.8799878,
 'total_2023_cny_100m': 2696.03008375,
 'regional_change_pct': 22.442190250510798}

In [3]:
key_counts = {
    'years': data['year'].nunique(),
    'counties': data['county'].nunique(),
    'services': data['service'].nunique(),
    'duplicate_keys': int(data.duplicated(['year', 'county', 'service']).sum()),
    'missing_values': int(data.isna().sum().sum()),
}
key_counts


{'years': 2,
 'counties': 11,
 'services': 5,
 'duplicate_keys': 0,
 'missing_values': 0}

In [4]:
totals = regional_totals(data)
totals.assign(
    rounded_cny_100m=totals['total_cny_100m'].round(2),
    rounded_cny_billion=totals['total_cny_billion'].round(2),
)


,year,total_cny_100m,total_cny_billion,rounded_cny_100m,rounded_cny_billion
0,2018,2201.879988,220.187999,2201.88,220.19
1,2023,2696.030084,269.603008,2696.03,269.60


The corrected 2023 total is 2,696.03 CNY 100 million (CNY 269.60 billion). All structural and unit checks must pass before the next notebooks run.